# Experiment Runner

Runs the configured search algorithm across all seeds, trains/evaluates each candidate, and logs everything to the SQLite DB, the result CSVs and MLflow.

> The model layers come from `src/models/architecture_builder.py`. A placeholder MLP ships so this runs today; replace `build_model` with your CNN/LSTM/GRU and re-run - nothing else changes.

In [ ]:
# Make the project root importable and the working directory (so relative paths
# like the sqlite mlflow uri resolve consistently with main.py).
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

## 1. Load configuration

The cell below loads `configs/` then applies a small **smoke override** so the notebook finishes in seconds. Comment out the `apply_overrides` block to run the full experiment from `experiment.yaml`.

In [ ]:
from src.utils.config import load_configs, apply_overrides
configs = load_configs()

# --- smoke override (delete/comment for the full run) ---------------
configs = apply_overrides(configs, {
    'experiment': {
        'seeds': [11, 22],
        'budget': {'evaluations': 5},
        'training': {'epochs': 1},
        'data': {'max_train_samples': 4000, 'max_val_samples': 2000, 'max_test_samples': 2000},
    }
})
# -------------------------------------------------------------------

print('algorithm:', configs['experiment']['algorithm'])
print('seeds    :', configs['experiment']['seeds'])
print('budget   :', configs['experiment']['budget'])
print('epochs   :', configs['experiment']['training']['epochs'])

## 1b. Load the fixed splits

The framework only **loads** the existing `data/splits/{train,val,test}_idx.npy` files — it never splits the dataset. You create them once yourself beforehand.

In [ ]:
# Load the fixed splits. The framework only LOADS existing split files and never
# splits the dataset itself. (Create data/splits/{train,val,test}_idx.npy once
# yourself beforehand.) Raises a clear error if they are missing.
from src.data import split_loader
train_idx, val_idx, test_idx = split_loader.load_splits()
print('split sizes -> train:', len(train_idx), 'val:', len(val_idx), 'test:', len(test_idx))

## 2. Run the experiment

`ExperimentManager.run()` performs the full sequence: setup dirs/DB/MLflow -> **load** the fixed splits (never regenerated) -> load data -> for each seed run the search (build, train, validate, log) -> export CSVs.

`device='auto'` uses the **GPU when available**, otherwise prints a warning and falls back to CPU.

In [ ]:
from src.experiment.experiment_manager import ExperimentManager
manager = ExperimentManager(configs=configs, device='auto')
result = manager.run()
print('\nExperiment id:', result['experiment_id'])

## 3. Inspect results from the database

In [ ]:
import pandas as pd
from src.tracking import database
runs = pd.DataFrame(database.fetch_all('runs'))
runs[['algorithm', 'seed', 'best_accuracy', 'best_precision', 'best_recall',
      'best_macro_f1', 'total_search_time', 'time_to_best', 'peak_memory_mb']]

In [ ]:
evals = pd.DataFrame(database.fetch_all('evaluations'))
evals[['seed', 'evaluation_number', 'accuracy', 'macro_f1',
       'training_time', 'peak_memory_mb', 'failed']].head(20)

In [ ]:
best = pd.DataFrame(database.fetch_all('best_models'))
best[['run_id', 'evaluation_number', 'accuracy', 'macro_f1', 'model_path']]

## 4. The exported CSVs (`results/csv/`)

In [ ]:
from src.utils import paths
print(paths.RUNS_CSV)
display(pd.read_csv(paths.RUNS_CSV))
print(paths.EVALUATIONS_CSV)
display(pd.read_csv(paths.EVALUATIONS_CSV).head())

## 5. View runs in MLflow

This framework logs to a **sqlite** backend, so the MLflow UI must be pointed at the *same* store. A bare `mlflow ui` uses the default `./mlruns` file store and will show **nothing**.

From a terminal at the project root:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

then open http://localhost:5000 (experiment **MITBIH_RS**).